# Install the visit-notes-to-invoice tool
Run the cell below once. It writes `visit_to_invoice.py` into the folder this notebook is in.
The tool is also saved in the `tools` folder of the repo.

In [ ]:
from pathlib import Path

Path('visit_to_invoice.py').write_text('#!/usr/bin/env python3\n"""Turn a week of visit notes into a weekly invoice.\n\nYou paste the visit notes (the ones written with docs/visit_note_form.md) into a\ntext file. This tool reads them, does the arithmetic with exact decimal math, shows\nyou what it understood, and, only after you say yes, writes a plain-text invoice you\ncan paste into an email. It can also add a row to the invoice tracker spreadsheet.\n\nThe money rules come from the client agreement (sections 5 to 8) and the README:\n  * Bill the actual time, arrival to departure, to the nearest minute.\n  * The shortest visit for that client still applies.\n  * Mileage is for drives with or for the client only, at the mileage rate.\n  * Purchases on the client\'s card are listed but not added.\n  * Purchases on my own card are added (backup only, under $100 each).\n  * Payment is due 7 days after the invoice is sent.\n\nNothing here is legal or tax advice. The tool does arithmetic; you check it.\nPrivate files (clients.json, notes, invoices) must stay out of the GitHub repo.\n\nUsage:\n  python visit_to_invoice.py notes.txt --client example --clients clients.json\n"""\n\nimport argparse\nimport json\nimport re\nimport sys\nfrom collections import defaultdict\nfrom dataclasses import dataclass, field\nfrom datetime import date, datetime, timedelta\nfrom decimal import Decimal, InvalidOperation, ROUND_HALF_UP\nfrom pathlib import Path\n\nCENT = Decimal("0.01")\nMAX_DAY_MINUTES = 12 * 60          # agreement: up to 12 hours in a day\nOWN_CARD_LIMIT = Decimal("100")    # agreement section 7: own card only under $100\nDEFAULT_MILEAGE_RATE = Decimal("0.76")  # IRS standard rate, July to December 2026\n\n\ndef money(x):\n    return Decimal(x).quantize(CENT, ROUND_HALF_UP)\n\n\ndef fmt(x):\n    return f"${money(x):,.2f}"\n\n\ndef hm(minutes):\n    h, m = divmod(int(minutes), 60)\n    return f"{h} h {m:02d} min"\n\n\ndef clock(minutes):\n    h, m = divmod(int(minutes), 60)\n    suffix = "am" if h < 12 else "pm"\n    return f"{(h % 12) or 12}:{m:02d} {suffix}"\n\n\ndef pretty_date(d):\n    return f"{d.strftime(\'%b\')} {d.day}, {d.year}"\n\n\n# --------------------------------------------------------------------------\n# Reading the notes\n# --------------------------------------------------------------------------\n\nLABELS = [\n    ("date", r"date"),\n    ("times", r"start time\\s*[-–—]\\s*end time|start\\s*/\\s*end time|times?"),\n    ("did", r"what we did"),\n    ("managed", r"what you managed on your own"),\n    ("noticed", r"anything i noticed"),\n    ("drives", r"drives or purchases[^:]*"),\n    ("next", r"next visit"),\n]\nLABEL_RES = [(k, re.compile(rf"^\\s*(?:{p})\\s*:\\s*(.*)$", re.I)) for k, p in LABELS]\nNOTE_START = re.compile(r"^\\s*visit note\\b", re.I)\nTIME_RE = re.compile(r"(\\d{1,2})(?::(\\d{2}))?\\s*([ap])\\.?\\s*m?\\.?", re.I)\nMILES_RE = re.compile(r"(\\d+(?:\\.\\d+)?)\\s*(?:miles?|mi)\\b", re.I)\nDOLLAR_RE = re.compile(r"\\$\\s*(\\d+(?:,\\d{3})*(?:\\.\\d{1,2})?)")\nSENTENCE_SPLIT = re.compile(r"\\.\\s+(?=[A-Z])|\\n|;\\s*")\nCLIENT_CARD = ("your card", "client\'s card", "client card", "their card",\n               "your debit", "your credit")\nMY_CARD = ("my card", "my own card", "own card", "my debit", "my credit")\n\n\n@dataclass\nclass Purchase:\n    date: date\n    text: str\n    amount: Decimal\n    card: str  # "client", "mine" or "unknown"\n\n\n@dataclass\nclass Visit:\n    day: date = None\n    start: int = None          # minutes after midnight\n    end: int = None\n    actual: int = None         # minutes worked\n    miles: Decimal = Decimal("0")\n    purchases: list = field(default_factory=list)\n    errors: list = field(default_factory=list)    # block writing the invoice\n    warnings: list = field(default_factory=list)  # shown, do not block\n    label: str = ""            # how to point at this note in messages\n\n\ndef parse_date(raw, year):\n    s = raw.strip()\n    weekday = None\n    m = re.match(r"^([A-Za-z]+day),?\\s*(.*)$", s)\n    if m:\n        weekday, s = m.group(1), m.group(2)\n    s = re.sub(r"(\\d)(st|nd|rd|th)\\b", r"\\1", s, flags=re.I)\n    s = s.replace(",", " ").replace("Sept ", "Sep ")\n    s = re.sub(r"\\s+", " ", s).strip()\n    tries = []\n    if re.search(r"\\b\\d{4}\\b", s) or re.search(r"\\d+/\\d+/\\d+", s):\n        tries += [(s, "%B %d %Y"), (s, "%b %d %Y"), (s, "%m/%d/%Y")]\n    else:\n        tries += [(f"{s} {year}", "%B %d %Y"), (f"{s} {year}", "%b %d %Y"),\n                  (f"{s}/{year}", "%m/%d/%Y")]\n    for text, f in tries:\n        try:\n            return datetime.strptime(text, f).date(), weekday\n        except ValueError:\n            continue\n    return None, weekday\n\n\ndef parse_times(raw):\n    found = []\n    for m in TIME_RE.finditer(raw):\n        h, mi, ap = int(m.group(1)), int(m.group(2) or 0), m.group(3).lower()\n        if not (1 <= h <= 12 and 0 <= mi < 60):\n            return None\n        found.append((h % 12) * 60 + mi + (720 if ap == "p" else 0))\n    return found if len(found) == 2 else None\n\n\ndef parse_drives(text, day):\n    miles_found = [Decimal(x) for x in MILES_RE.findall(text)]\n    purchases, notes = [], []\n    for sentence in SENTENCE_SPLIT.split(text):\n        s = sentence.strip(" .\\t")\n        amounts = DOLLAR_RE.findall(s)\n        if not amounts:\n            continue\n        low = s.lower()\n        is_client = any(w in low for w in CLIENT_CARD)\n        is_mine = any(w in low for w in MY_CARD)\n        card = "client" if is_client and not is_mine else "mine" if is_mine and not is_client else "unknown"\n        if card == "unknown" and re.search(r"\\bmiles?\\b", low):\n            notes.append(f"Ignored the dollar amount in a mileage sentence: \\"{s}\\"")\n            continue\n        for a in amounts:\n            try:\n                purchases.append(Purchase(day, s, Decimal(a.replace(",", "")), card))\n            except InvalidOperation:\n                pass\n    return miles_found, purchases, notes\n\n\ndef parse_notes(text, year):\n    """Return a list of Visit objects, one per \'Visit note\' block."""\n    blocks, current = [], None\n    for line in text.splitlines():\n        if NOTE_START.match(line):\n            current = []\n            blocks.append(current)\n        elif current is not None:\n            current.append(line)\n    visits = []\n    for i, lines in enumerate(blocks, 1):\n        fields, key = defaultdict(list), None\n        for line in lines:\n            for k, rx in LABEL_RES:\n                m = rx.match(line)\n                if m:\n                    key = k\n                    fields[k].append(m.group(1))\n                    break\n            else:\n                if key and line.strip():\n                    fields[key].append(line.strip())\n        v = Visit(label=f"note {i}")\n        joined = {k: " ".join(x for x in vals if x).strip() for k, vals in fields.items()}\n\n        day, weekday = parse_date(joined.get("date", ""), year) if joined.get("date") else (None, None)\n        if day is None:\n            v.errors.append("I could not read the date. Write it like: Tuesday, October 13")\n        else:\n            v.day = day\n            v.label = f"note {i} ({pretty_date(day)})"\n            if weekday and day.strftime("%A").lower() != weekday.lower():\n                v.warnings.append(\n                    f"{weekday} {day.strftime(\'%B\')} {day.day} is not a {weekday} in {day.year}. "\n                    "Check the date, or run with --year if it is another year.")\n\n        times = parse_times(joined.get("times", "")) if joined.get("times") else None\n        if times is None:\n            v.errors.append("I could not read the start and end times. "\n                            "Write them like: 10:00 am - 2:15 pm (with am or pm on both).")\n        else:\n            v.start, v.end = times\n            if v.end <= v.start:\n                v.errors.append("The end time is not after the start time. Check am and pm.")\n            else:\n                v.actual = v.end - v.start\n\n        drives = joined.get("drives", "")\n        if drives:\n            miles_found, v.purchases, notes = parse_drives(drives, day)\n            v.miles = sum(miles_found, Decimal("0"))\n            v.warnings.extend(notes)\n            if len(miles_found) > 1:\n                parts = " + ".join(str(x) for x in miles_found)\n                v.warnings.append(f"Found {len(miles_found)} mileage numbers ({parts}). "\n                                  "Check none is counted twice.")\n            if re.search(r"\\bmiles?\\b", drives, re.I) and not miles_found:\n                v.warnings.append("The note mentions miles but I found no number.")\n        visits.append(v)\n\n    seen = {}\n    for v in visits:\n        if v.day and v.start is not None:\n            k = (v.day, v.start, v.end)\n            if k in seen:\n                v.warnings.append(f"Looks like a repeat of {seen[k]}. Remove one if it is.")\n            seen[k] = v.label\n    return visits\n\n\n# --------------------------------------------------------------------------\n# The money\n# --------------------------------------------------------------------------\n\n@dataclass\nclass Line:\n    day: date\n    start: int\n    end: int\n    actual: int\n    billed: int\n    amount: Decimal\n    miles: Decimal\n    mileage_amount: Decimal\n\n\n@dataclass\nclass Invoice:\n    number: str\n    client_key: str\n    client_name: str\n    payer: str\n    sent: date\n    due: date\n    first: date\n    last: date\n    rate: Decimal\n    min_hours: Decimal\n    mileage_rate: Decimal\n    lines: list\n    purchases: list\n    visits_total: Decimal\n    mileage_total: Decimal\n    repay_total: Decimal\n    total: Decimal\n    warnings: list\n    unsure: list\n\n\ndef compute_invoice(visits, client, client_key, number, sent):\n    rate = Decimal(str(client["rate"]))\n    min_hours = Decimal(str(client["minimum_hours"]))\n    mileage_rate = Decimal(str(client.get("mileage_rate", DEFAULT_MILEAGE_RATE)))\n    min_minutes = int((min_hours * 60).to_integral_value(ROUND_HALF_UP))\n\n    lines, purchases, warnings = [], [], []\n    for v in sorted((x for x in visits if not x.errors), key=lambda x: (x.day, x.start)):\n        billed = max(v.actual, min_minutes)\n        amount = money(Decimal(billed) * rate / 60)\n        mileage_amount = money(v.miles * mileage_rate)\n        lines.append(Line(v.day, v.start, v.end, v.actual, billed, amount, v.miles, mileage_amount))\n        purchases.extend(v.purchases)\n        for w in v.warnings:\n            warnings.append(f"{v.label}: {w}")\n    if not lines:\n        raise ValueError("No visits could be billed.")\n\n    per_day = defaultdict(int)\n    for ln in lines:\n        per_day[ln.day] += ln.billed\n    for d, mins in per_day.items():\n        if mins > MAX_DAY_MINUTES:\n            warnings.append(f"{pretty_date(d)}: {hm(mins)} billed in one day is over the "\n                            "12 hours a day the agreement allows.")\n\n    mine = [p for p in purchases if p.card == "mine"]\n    for p in mine:\n        if p.amount >= OWN_CARD_LIMIT:\n            warnings.append(f"{pretty_date(p.date)}: {fmt(p.amount)} on my own card is not "\n                            "under $100. The agreement says my card is a backup only under $100.")\n    unsure = [p for p in purchases if p.card == "unknown"]\n\n    visits_total = sum((ln.amount for ln in lines), Decimal("0"))\n    mileage_total = sum((ln.mileage_amount for ln in lines), Decimal("0"))\n    repay_total = sum((p.amount for p in mine), Decimal("0"))\n    return Invoice(\n        number=str(number), client_key=client_key, client_name=client["name"],\n        payer=client.get("payer", ""), sent=sent, due=sent + timedelta(days=7),\n        first=lines[0].day, last=lines[-1].day, rate=rate, min_hours=min_hours,\n        mileage_rate=mileage_rate, lines=lines, purchases=purchases,\n        visits_total=money(visits_total), mileage_total=money(mileage_total),\n        repay_total=money(repay_total),\n        total=money(visits_total + mileage_total + repay_total),\n        warnings=warnings, unsure=unsure)\n\n\n# --------------------------------------------------------------------------\n# Showing the result\n# --------------------------------------------------------------------------\n\ndef render_understood(inv, visits):\n    out = ["WHAT I UNDERSTOOD (check this before saying yes)", ""]\n    for ln in inv.lines:\n        short = "  (shortest visit applies)" if ln.billed > ln.actual else ""\n        out.append(f"{pretty_date(ln.day)}  {clock(ln.start)} - {clock(ln.end)}  "\n                   f"worked {hm(ln.actual)}  billed {hm(ln.billed)}  {fmt(ln.amount)}{short}")\n        if ln.miles:\n            out.append(f"    miles {ln.miles} x {fmt(inv.mileage_rate)} = {fmt(ln.mileage_amount)}")\n    if inv.purchases:\n        out.append("")\n        out.append("Purchases found:")\n    for p in inv.purchases:\n        where = {"client": "client\'s card, listed only, not added",\n                 "mine": "my card, added to the total",\n                 "unknown": "NOT ADDED: I could not tell whose card"}[p.card]\n        out.append(f"    {pretty_date(p.date)}  {fmt(p.amount)} ({where}): {p.text}")\n    out += ["", f"Visits {fmt(inv.visits_total)}   Mileage {fmt(inv.mileage_total)}   "\n                f"My card to be repaid {fmt(inv.repay_total)}",\n            f"TOTAL DUE {fmt(inv.total)}   (due {pretty_date(inv.due)})"]\n    problems = [(v.label, e) for v in visits for e in v.errors]\n    if problems:\n        out += ["", "PROBLEMS (these visits are NOT in the invoice; fix the note and run again):"]\n        out += [f"  - {lab}: {e}" for lab, e in problems]\n    if inv.unsure:\n        out += ["", "NEEDS YOUR DECISION (not added to the total):"]\n        out += [f"  - {fmt(p.amount)} on {pretty_date(p.date)}: \\"{p.text}\\". "\n                "Say \\"your card\\" or \\"my card\\" in the note." for p in inv.unsure]\n    if inv.warnings:\n        out += ["", "CHECK THESE:"] + [f"  - {w}" for w in inv.warnings]\n    return "\\n".join(out)\n\n\ndef render_invoice(inv, config):\n    sender = config.get("from", "[Your full name], [Last Name] Home Support")\n    L = [f"INVOICE {inv.number} - {sender.split(\',\', 1)[-1].strip() if \',\' in sender else sender}",\n         f"From: {sender}"]\n    if config.get("contact"):\n        L.append(f"Contact: {config[\'contact\']}")\n    L += [f"Client: {inv.client_name}"]\n    if inv.payer:\n        L.append(f"Paid by: {inv.payer}")\n    week = (pretty_date(inv.first) if inv.first == inv.last\n            else f"{pretty_date(inv.first)} to {pretty_date(inv.last)}")\n    L += [f"Date sent: {pretty_date(inv.sent)}",\n          f"Week covered: {week}",\n          f"Payment due: {pretty_date(inv.due)} (within 7 days)",\n          f"Hourly rate: {fmt(inv.rate)} an hour",\n          f"Shortest visit: {inv.min_hours.normalize():f} hours", "",\n          "VISITS (actual time worked, to the nearest minute; the shortest visit applies)",\n          f"{\'Date\':<14}{\'Time\':<22}{\'Billed\':<12}{\'Amount\':>10}"]\n    for ln in inv.lines:\n        L.append(f"{pretty_date(ln.day):<14}{clock(ln.start) + \' - \' + clock(ln.end):<22}"\n                 f"{hm(ln.billed):<12}{fmt(ln.amount):>10}")\n    L += [f"{\'Visits total\':<48}{fmt(inv.visits_total):>10}", ""]\n\n    drives = [ln for ln in inv.lines if ln.miles]\n    if drives:\n        L += [f"MILEAGE (drives with or for the client, {fmt(inv.mileage_rate)} a mile)",\n              f"{\'Date\':<14}{\'Miles\':<10}{\'Amount\':>10}"]\n        for ln in drives:\n            L.append(f"{pretty_date(ln.day):<14}{str(ln.miles):<10}{fmt(ln.mileage_amount):>10}")\n        L += [f"{\'Mileage total\':<24}{fmt(inv.mileage_total):>10}", ""]\n\n    listed = [p for p in inv.purchases if p.card in ("client", "mine")]\n    if listed:\n        L += ["PURCHASES (receipts attached; the client\'s card is listed for the record only)"]\n        for p in listed:\n            card = "client\'s card" if p.card == "client" else "my card"\n            L.append(f"{pretty_date(p.date)}  {fmt(p.amount):>9}  {card:<14}{p.text}")\n        L += [f"{\'On my card, to be repaid\':<24}{fmt(inv.repay_total):>10}", ""]\n\n    L += ["TOTAL DUE",\n          f"{\'Visits\':<34}{fmt(inv.visits_total):>10}",\n          f"{\'Mileage\':<34}{fmt(inv.mileage_total):>10}",\n          f"{\'Purchases on my card\':<34}{fmt(inv.repay_total):>10}",\n          f"{\'TOTAL DUE THIS WEEK\':<34}{fmt(inv.total):>10}",\n          f"Please pay by {pretty_date(inv.due)}, by check, cash or Zelle.",\n          "I give a receipt for every cash payment and every reimbursement."]\n    return "\\n".join(L) + "\\n"\n\n\n# --------------------------------------------------------------------------\n# Tracker spreadsheet\n# --------------------------------------------------------------------------\n\nTRACKER_FIRST_ROW, TRACKER_LAST_ROW = 10, 59   # row 9 of the tracker is the example\n\n\ndef tracker_next_number(path):\n    from openpyxl import load_workbook\n    ws = load_workbook(path).active\n    used = sum(1 for r in range(TRACKER_FIRST_ROW, TRACKER_LAST_ROW + 1) if ws.cell(r, 1).value)\n    return used + 1\n\n\ndef add_to_tracker(path, inv):\n    """Fill the next empty tracker row. Formulas in the grey columns are left alone."""\n    from openpyxl import load_workbook\n    wb = load_workbook(path)\n    ws = wb.active\n    for r in range(TRACKER_FIRST_ROW, TRACKER_LAST_ROW + 1):\n        if not ws.cell(r, 1).value:\n            week = (f"{inv.first.strftime(\'%b\')} {inv.first.day}-{inv.last.day}"\n                    if (inv.first.month == inv.last.month) else\n                    f"{inv.first.strftime(\'%b\')} {inv.first.day} - {inv.last.strftime(\'%b\')} {inv.last.day}")\n            ws.cell(r, 1).value = inv.number\n            ws.cell(r, 2).value = inv.client_name\n            ws.cell(r, 3).value = week\n            ws.cell(r, 4).value = inv.sent\n            ws.cell(r, 6).value = float(inv.total)\n            wb.save(path)\n            return r\n    raise RuntimeError("The tracker has no empty rows left.")\n\n\n# --------------------------------------------------------------------------\n# Command line\n# --------------------------------------------------------------------------\n\ndef load_config(path):\n    with open(path, encoding="utf-8") as f:\n        return json.load(f)\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser(description="Turn visit notes into a weekly invoice.")\n    ap.add_argument("notes", help="text file with the pasted visit notes for one client")\n    ap.add_argument("--client", required=True, help="client key from clients.json")\n    ap.add_argument("--clients", default="clients.json", help="path to clients.json (private)")\n    ap.add_argument("--year", type=int, default=date.today().year,\n                    help="year for dates written without one (default: this year)")\n    ap.add_argument("--sent", help="date the invoice is sent, YYYY-MM-DD (default: today)")\n    ap.add_argument("--number", help="invoice number (default: next one in the tracker, or 1)")\n    ap.add_argument("--tracker", help="path to invoice_tracker.xlsx to add a row to")\n    ap.add_argument("--outdir", default="invoices", help="folder for the invoice (default: invoices)")\n    ap.add_argument("--yes", action="store_true", help="skip the question and write the files")\n    args = ap.parse_args(argv)\n\n    config = load_config(args.clients)\n    clients = config.get("clients", {})\n    if args.client not in clients:\n        sys.exit(f"No client \'{args.client}\' in {args.clients}. Known: {\', \'.join(clients) or \'none\'}")\n    client = clients[args.client]\n\n    text = Path(args.notes).read_text(encoding="utf-8", errors="replace")\n    visits = parse_notes(text, args.year)\n    if not visits:\n        sys.exit("I found no \'Visit note\' in that file.")\n\n    sent = datetime.strptime(args.sent, "%Y-%m-%d").date() if args.sent else date.today()\n    number = args.number\n    if number is None:\n        number = tracker_next_number(args.tracker) if args.tracker else 1\n    try:\n        inv = compute_invoice(visits, client, args.client, number, sent)\n    except ValueError as e:\n        print(render_understood_errors(visits))\n        sys.exit(str(e))\n\n    print(render_understood(inv, visits))\n    blocked = any(v.errors for v in visits)\n    if blocked:\n        print("\\nNothing was written, because some visits could not be read.")\n        return 1\n    if not args.yes:\n        if input("\\nDoes this look right? Write the invoice? [y/N] ").strip().lower() != "y":\n            print("Nothing was written.")\n            return 0\n\n    outdir = Path(args.outdir)\n    outdir.mkdir(parents=True, exist_ok=True)\n    out = outdir / f"invoice_{args.client}_{inv.number}_{sent.isoformat()}.txt"\n    if out.exists():\n        sys.exit(f"{out} already exists. Nothing was written. Use --number to pick another.")\n    out.write_text(render_invoice(inv, config), encoding="utf-8")\n    print(f"\\nWrote {out}")\n    if args.tracker:\n        row = add_to_tracker(args.tracker, inv)\n        print(f"Added invoice {inv.number} to row {row} of {args.tracker}")\n    print("Reminder: attach the receipts, and keep this folder out of GitHub.")\n    return 0\n\n\ndef render_understood_errors(visits):\n    return "\\n".join(f"{v.label}: {e}" for v in visits for e in v.errors)\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n', encoding='utf-8')
print('Wrote visit_to_invoice.py')
